# FuzzyDiff Table 35: ablations, one session

Settings: GPU T4 x2, Internet on, Kaggle Secret `HF_TOKEN` (account that accepted the SD3-Medium licence).

The table is split over 4 sessions: set `SUITE` and `PART` in section 1 and run the notebook top to bottom.

| Session | SUITE | PART | Arms |
|---|---|---|---|
| 1 | mixture | A | SD3, Stage 1 and its ablations |
| 2 | mixture | B | Stage 2 arms (with `unguided`, `stage1`, `stage1_off`) |
| 3 | size | A | as session 1 |
| 4 | size | B | as session 2 |

Prompts come from the dev group, so the numbers are a pilot under the provisional calibration, not paper evidence. Finished images are skipped on a rerun, so a session that times out is continued by running the same cells again.

In [ ]:
BRANCH = 'fuzzysd3'
OUT = '/kaggle/working/outputs'
VAE_DEVICE = 'cuda:1'  # the transformer stays on cuda:0

!git clone -q --branch {BRANCH} https://github.com/sunzidulislam/fuzzydiff.git /kaggle/working/fuzzydiff
%cd /kaggle/working/fuzzydiff
!pip install -q -r requirements-kaggle.txt
# CLIP-IQA only; --no-deps because piq would pull in a torchvision that does not match this torch.
!pip install -q --no-deps piq==0.8.0
# Kaggle's torchvision is built for its own torch and crashes on import with this one.
!pip uninstall -y -q torchvision
!python -I -c "from transformers import CLIPSegForImageSegmentation, SiglipModel; from diffusers import StableDiffusion3Pipeline; print('libraries ok')"
!git log -1 --oneline

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi -L

## 1. Choose the session and the arms

`SMOKE = True` runs 2 prompts per setting to catch typos and to time one image; set it to `False` for the real run (30 prompts).

In [ ]:
SUITE, PART = 'mixture', 'A'     # session 1: mixture/A, 2: mixture/B, 3: size/A, 4: size/B
SMOKE = True
GROUP, N = 'dev', (2 if SMOKE else 30)
FLAGS = '--sigma-hi 0.9 --clip rms --dmax 0.02 --eta0 0.04' if SUITE == 'size' else ''

A_T1 = ['unguided', 'stage1', 'abl_attention_only', 'abl_overlap', 'abl_floored', 'abl_weighted_sum',
        'abl_no_backtrack', 'abl_hard_domain', 'abl_no_satisfaction', 'abl_hard_interval', 'abl_no_presence',
        'abl_cond_branch', 'abl_global_presence', 'abl_image_projection', 'abl_s1_extra']
A_T2 = ['stage1', 'abl_point', 'abl_one_sided', 'abl_cond_x0', 'abl_no_acceptance', 'abl_accept_refreeze',
        'abl_accept_no_tier2', 'abl_linear_domain', 'abl_no_tau_min', 'abl_no_core'] + \
       (['abl_size_no_margin'] if SUITE == 'size' else [])
B_T1 = ['unguided', 'stage1', 'stage1_off', 'stage1_plain', 'stage1_sar', 'stage1_placebo', 'stage1_plain_tm',
        'stage1_sar_attn_prior', 'stage1_sar_post', 'sd3_sar']
B_T2 = ['stage1', 'stage1_sar', 'stage1_sar_conf', 'stage1_sar_comp', 'stage1_sar_comp_all', 'stage1_sar_nocheck',
        'stage1_sar_drift_prior', 'stage1_sar_lagged']
T1, T2 = (A_T1, A_T2) if PART == 'A' else (B_T1, B_T2)
DEST = f'{OUT}/abl35/{SUITE}'
print(SUITE, PART, '|', len(T1), 'must arms,', len(T2), 'extra arms,', N, 'prompts each')

## 2. Restore an earlier session (only when continuing one)

Attach the earlier `abl35_*_*.zip` as a Kaggle Dataset and put its path below. Leave it empty on a fresh start.

In [ ]:
RESUME_ZIP = ''   # e.g. '/kaggle/input/abl35-mixture-a/abl35_mixture_A.zip'
if RESUME_ZIP:
    !mkdir -p {OUT}/abl35 && unzip -q -n {RESUME_ZIP} -d {OUT}/abl35
    !ls {DEST}

## 3. Must arms

In [ ]:
ARMS = ','.join(T1)
!timeout 11h python -I code/run_experiment.py --suite {SUITE} --group {GROUP} --n {N} --arms {ARMS} {FLAGS} --out {DEST} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -3

In [ ]:
# Check: per-arm image count, failures and time per image. Look for a Traceback or a failed arm.
import glob, json, collections
n, failed, t = collections.Counter(), collections.defaultdict(int), collections.defaultdict(list)
for p in glob.glob(f'{DEST}/*/*/meta.json'):
    m = json.load(open(p))
    n[m['arm']] += 1
    failed[m['arm']] += bool(m.get('failed'))
    if m.get('time_s'):
        t[m['arm']].append(m['time_s'])
for a in sorted(n):
    print(f"{a:26s} images {n[a]:3d}  failed {failed[a]}  mean time_s {sum(t[a]) / max(len(t[a]), 1):7.1f}")

## 4. Extra arms

Run only when time is left in the session (about 11 h of GPU per session is the limit).

In [ ]:
ARMS = ','.join(T2)
!timeout 11h python -I code/run_experiment.py --suite {SUITE} --group {GROUP} --n {N} --arms {ARMS} {FLAGS} --out {DEST} --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -3

## 5. Save the outputs

Run this at the end of every session and download the zip: Kaggle removes the working folder afterwards.

In [ ]:
!cd {OUT}/abl35 && zip -qr ../abl35_{SUITE}_{PART}.zip {SUITE} && ls -la ../abl35_{SUITE}_{PART}.zip

## 6. After all four sessions: rating page and score

Unzip the A and B zips of a setting into one `{OUT}/abl35/{SUITE}` folder (where `stage1` and `unguided` exist in both, keep part A's copy), attach them as a Dataset or upload them, and run the two cells below in a new session. Rate the page blind, with at least two raters and the rubric of Tables 1-2. Ratings by an AI model are a pilot and must be labelled that way.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, 'code')
from annotate import ABLATION_ARM_NAMES, ABLATION_COMPARISONS, build_mixture_page, build_size_page
done = {p.name for p in Path(DEST).iterdir() if p.is_dir()}
arms = [a for a in ABLATION_ARM_NAMES if a in done]
build = build_mixture_page if SUITE == 'mixture' else build_size_page
build(DEST, f'{OUT}/abl35/{SUITE}.html', 0, arms=arms)
!cd {OUT}/abl35 && zip -q ../abl35_{SUITE}_page.zip {SUITE}.html {SUITE}.key.json && ls -la ../abl35_{SUITE}_page.zip

In [ ]:
RATINGS = r'''PASTE HERE'''
from annotate import ablation_table_rows, score_mixture
Path(f'{OUT}/abl35/{SUITE}.ratings.json').write_text(RATINGS)
comps = [(a, b) for a, b in ABLATION_COMPARISONS if a in arms and b in arms]
s = score_mixture(f'{OUT}/abl35/{SUITE}.ratings.json', f'{OUT}/abl35/{SUITE}.key.json',
                  field='leakage' if SUITE == 'mixture' else 'size', comparisons=comps)
print(ablation_table_rows(**{SUITE: s}))